# Yield Analysis Notebook

This notebook follows a typical project workflow:
1. Load and check input data
2. Build paired differences (`heat - control`)
3. Show statistical summaries and model results
4. Preview report content
5. Save outputs in the final cell


In [1]:
from datetime import datetime
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

ROOT = Path.cwd().resolve()
if not (ROOT / "cleaned_data").exists() and (ROOT.parent / "cleaned_data").exists():
    ROOT = ROOT.parent

CLEAN_DIR = ROOT / "cleaned_data"
RESULT_DIR = ROOT / "results" / "yield"

LT_PATH = CLEAN_DIR / "longterm_yield_analysis_locked.csv"
ACUTE_PATH = CLEAN_DIR / "acute_yield_analysis_locked.csv"

LT_SUMMARY_PATH = RESULT_DIR / "yield_longterm_summary.csv"
LT_COEF_PATH = RESULT_DIR / "yield_longterm_mixedlm_coefficients.csv"
LT_WALD_PATH = RESULT_DIR / "yield_longterm_mixedlm_wald_terms.csv"

ACUTE_SUMMARY_PATH = RESULT_DIR / "yield_acute_summary.csv"
ACUTE_COEF_PATH = RESULT_DIR / "yield_acute_mixedlm_coefficients.csv"
ACUTE_WALD_PATH = RESULT_DIR / "yield_acute_mixedlm_wald_terms.csv"

REPORT_PATH = RESULT_DIR / "yield_analysis_report.md"

OUTCOMES = ["healthy_weight_g", "pct_rotten_count", "pct_rotten_weight"]

print("ROOT:", ROOT)
print("LT_PATH exists:", LT_PATH.exists())
print("ACUTE_PATH exists:", ACUTE_PATH.exists())


ROOT: /Users/liyuang/Desktop/STAT628/installment3
LT_PATH exists: True
ACUTE_PATH exists: True


In [2]:
def paired_ttest_table(df: pd.DataFrame, group_cols: list[str], outcomes: list[str]) -> pd.DataFrame:
    rows = []
    grouped = [(("overall",), df)] if not group_cols else list(df.groupby(group_cols))
    for key, sub in grouped:
        if not isinstance(key, tuple):
            key = (key,)
        for outcome in outcomes:
            vals = sub[outcome].dropna().to_numpy()
            if len(vals) < 2:
                t_stat = np.nan
                p_value = np.nan
                se = np.nan
            else:
                t_stat, p_value = stats.ttest_1samp(vals, 0.0)
                se = vals.std(ddof=1) / np.sqrt(len(vals))
            row = {
                "outcome": outcome,
                "n_pairs": int(len(vals)),
                "mean_diff_heat_minus_control": float(np.nanmean(vals)) if len(vals) else np.nan,
                "sd_diff": float(np.nanstd(vals, ddof=1)) if len(vals) > 1 else np.nan,
                "se_diff": se,
                "t_stat": t_stat,
                "p_value": p_value,
            }
            for col, value in zip(group_cols, key):
                row[col] = value
            rows.append(row)
    cols = group_cols + [
        "outcome",
        "n_pairs",
        "mean_diff_heat_minus_control",
        "sd_diff",
        "se_diff",
        "t_stat",
        "p_value",
    ]
    return pd.DataFrame(rows)[cols]


def fit_mixedlm(formula: str, data: pd.DataFrame, group_col: str):
    methods = ["lbfgs", "powell", "cg", "nm"]
    fallback = None
    errors = []
    for method in methods:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                model = smf.mixedlm(formula, data=data, groups=data[group_col]).fit(
                    reml=False,
                    method=method,
                    maxiter=1000,
                    disp=False,
                )
            if model.converged:
                return model, method
            if fallback is None:
                fallback = (model, method)
        except Exception as exc:
            errors.append(f"{method}: {exc}")
    if fallback is not None:
        return fallback
    raise RuntimeError("MixedLM failed for all optimizers: " + " | ".join(errors))


def mixedlm_coefficient_table(model, outcome: str, model_name: str, optimizer: str) -> pd.DataFrame:
    ci = model.conf_int()
    return pd.DataFrame(
        {
            "model_name": model_name,
            "optimizer": optimizer,
            "outcome": outcome,
            "term": model.params.index,
            "estimate": model.params.values,
            "std_error": model.bse.values,
            "z_value": model.tvalues.values,
            "p_value": model.pvalues.values,
            "ci_low": ci[0].values,
            "ci_high": ci[1].values,
            "n_obs": int(model.nobs),
            "n_groups": int(len(model.model.group_labels)),
            "converged": bool(model.converged),
            "log_likelihood": float(model.llf),
            "aic": float(model.aic),
            "bic": float(model.bic),
        }
    )


def _scalar(value) -> float:
    arr = np.asarray(value).astype(float).reshape(-1)
    return float(arr[0]) if arr.size else np.nan


def mixedlm_wald_terms_table(model, outcome: str, model_name: str) -> pd.DataFrame:
    try:
        table = model.wald_test_terms(skip_single=False).table.reset_index().rename(columns={"index": "term"})
        p_col = "pvalue" if "pvalue" in table.columns else "p_value"
        out = pd.DataFrame(
            {
                "model_name": model_name,
                "outcome": outcome,
                "term": table["term"],
                "statistic": table["statistic"].map(_scalar),
                "df_constraint": table["df_constraint"].map(_scalar),
                "p_value": table[p_col].map(_scalar),
            }
        )
        return out
    except Exception as exc:
        return pd.DataFrame(
            [
                {
                    "model_name": model_name,
                    "outcome": outcome,
                    "term": "<wald_test_failed>",
                    "statistic": np.nan,
                    "df_constraint": np.nan,
                    "p_value": np.nan,
                    "error": str(exc),
                }
            ]
        )


def build_longterm_pairs(df: pd.DataFrame) -> pd.DataFrame:
    keep = df[df["include_in_yield_model"] == 1].copy()
    idx = ["cultivar", "set_id"]
    values = ["healthy_weight_g", "pct_rotten_count", "pct_rotten_weight", "plot_id"]
    wide = keep.pivot(index=idx, columns="heat_trt", values=values)
    wide.columns = [f"{v}_{k.lower()}" for v, k in wide.columns]
    wide = wide.reset_index()
    wide["healthy_weight_diff"] = wide["healthy_weight_g_otc"] - wide["healthy_weight_g_control"]
    wide["pct_rotten_count_diff"] = wide["pct_rotten_count_otc"] - wide["pct_rotten_count_control"]
    wide["pct_rotten_weight_diff"] = wide["pct_rotten_weight_otc"] - wide["pct_rotten_weight_control"]
    return wide


def build_acute_pairs(df: pd.DataFrame) -> pd.DataFrame:
    keep = df[df["include_in_yield_model"] == 1].copy()
    idx = ["cultivar", "heat_level", "replicate"]
    values = [
        "healthy_weight_g",
        "pct_rotten_count",
        "pct_rotten_weight",
        "include_in_temp_model",
        "sensor_missing_overview",
        "sensor_missing_files",
        "sensor_missing_window",
    ]
    wide = keep.pivot(index=idx, columns="is_control", values=values)
    wide.columns = [
        f"{v}_{'control' if int(k) == 1 else 'heat'}" for v, k in wide.columns
    ]
    wide = wide.reset_index()
    wide["healthy_weight_diff"] = wide["healthy_weight_g_heat"] - wide["healthy_weight_g_control"]
    wide["pct_rotten_count_diff"] = wide["pct_rotten_count_heat"] - wide["pct_rotten_count_control"]
    wide["pct_rotten_weight_diff"] = wide["pct_rotten_weight_heat"] - wide["pct_rotten_weight_control"]
    wide["temp_model_usable_pair"] = (
        (wide["include_in_temp_model_heat"] == 1)
        & (wide["include_in_temp_model_control"] == 1)
    ).astype(int)
    return wide


def _best_term_lines(term_df: pd.DataFrame, heading: str) -> list[str]:
    lines = [heading]
    for outcome in OUTCOMES:
        sub = term_df[(term_df["outcome"] == outcome) & (term_df["term"] != "Intercept")].copy()
        sub = sub.dropna(subset=["p_value"])
        if sub.empty:
            lines.append(f"  - {outcome}: no valid Wald term test")
            continue
        best = sub.sort_values("p_value", ascending=True).iloc[0]
        lines.append(
            f"  - {outcome}: smallest term p={best['p_value']:.4g} ({best['term']})"
        )
    return lines


def build_report_lines(
    lt: pd.DataFrame,
    acute: pd.DataFrame,
    lt_pairs: pd.DataFrame,
    acute_pairs: pd.DataFrame,
    lt_summary: pd.DataFrame,
    acute_summary: pd.DataFrame,
    lt_terms: pd.DataFrame,
    acute_terms: pd.DataFrame,
) -> list[str]:
    lt_hw = lt_summary[lt_summary["outcome"] == "healthy_weight_g"].copy()
    lt_rot = lt_summary[lt_summary["outcome"] == "pct_rotten_count"].copy()
    acute_hw = acute_summary[acute_summary["outcome"] == "healthy_weight_g"].copy()
    acute_rot = acute_summary[acute_summary["outcome"] == "pct_rotten_count"].copy()

    report_lines = [
        "# Yield Analysis Report",
        "",
        f"- Generated at: {datetime.now().isoformat(timespec='seconds')}",
        "- Outcome coding for paired summaries: all differences are `heat - control`.",
        "- Primary model: mixed-effects model on row-level data.",
        "- Long-term mixed model: fixed `cultivar * heat_trt`, random intercept `(1|cultivar:set_id)`.",
        "- Acute mixed model: fixed `cultivar * heat_level * is_control`, random intercept `(1|cultivar:heat_level:replicate)`.",
        "",
        "## Inputs",
        f"- {LT_PATH}",
        f"- {ACUTE_PATH}",
        "",
        "## Key counts",
        f"- Long-term rows in yield model: {int(lt['include_in_yield_model'].sum())}",
        f"- Long-term pairs: {len(lt_pairs)}",
        f"- Acute rows in yield model: {int(acute['include_in_yield_model'].sum())}",
        f"- Acute pairs: {len(acute_pairs)}",
        f"- Acute pairs usable for later temperature models: {int(acute_pairs['temp_model_usable_pair'].sum())}",
        "",
        "## Paired-summary quick check",
        "- Long-term healthy weight summary:",
    ]

    for _, row in lt_hw.fillna("").iterrows():
        label = row["cultivar"] if row.get("cultivar", "") else "overall"
        report_lines.append(
            f"  - {label}: mean diff={row['mean_diff_heat_minus_control']:.3f}, p={row['p_value'] if row['p_value'] != '' else np.nan}"
        )

    report_lines.append("- Long-term rotten-count proportion summary:")
    for _, row in lt_rot.fillna("").iterrows():
        label = row["cultivar"] if row.get("cultivar", "") else "overall"
        report_lines.append(
            f"  - {label}: mean diff={row['mean_diff_heat_minus_control']:.5f}, p={row['p_value'] if row['p_value'] != '' else np.nan}"
        )

    report_lines.append("- Acute healthy weight summary by heat level:")
    for _, row in acute_hw.fillna("").iterrows():
        label = (
            f"{row['cultivar']}-{row['heat_level']}"
            if row.get("cultivar", "") and row.get("heat_level", "")
            else row.get("heat_level", "overall")
        )
        report_lines.append(
            f"  - {label}: mean diff={row['mean_diff_heat_minus_control']:.3f}, p={row['p_value'] if row['p_value'] != '' else np.nan}"
        )

    report_lines.append("- Acute rotten-count proportion summary by heat level:")
    for _, row in acute_rot.fillna("").iterrows():
        label = (
            f"{row['cultivar']}-{row['heat_level']}"
            if row.get("cultivar", "") and row.get("heat_level", "")
            else row.get("heat_level", "overall")
        )
        report_lines.append(
            f"  - {label}: mean diff={row['mean_diff_heat_minus_control']:.5f}, p={row['p_value'] if row['p_value'] != '' else np.nan}"
        )

    report_lines.extend(["", "## Mixed-model term tests"])
    report_lines.extend(_best_term_lines(lt_terms, "- Long-term smallest p by outcome:"))
    report_lines.extend(_best_term_lines(acute_terms, "- Acute smallest p by outcome:"))

    report_lines.extend(
        [
            "",
            "## Outputs",
            f"- {LT_SUMMARY_PATH}",
            f"- {LT_COEF_PATH}",
            f"- {LT_WALD_PATH}",
            f"- {ACUTE_SUMMARY_PATH}",
            f"- {ACUTE_COEF_PATH}",
            f"- {ACUTE_WALD_PATH}",
            f"- {REPORT_PATH}",
            "",
        ]
    )
    return report_lines


## 1) Load and inspect locked input data


In [3]:
lt = pd.read_csv(LT_PATH)
acute = pd.read_csv(ACUTE_PATH)

lt["include_in_yield_model"] = lt["include_in_yield_model"].astype(int)
lt["include_in_temp_model"] = lt["include_in_temp_model"].astype(int)

acute["include_in_yield_model"] = acute["include_in_yield_model"].astype(int)
acute["include_in_temp_model"] = acute["include_in_temp_model"].astype(int)
acute["sensor_missing_overview"] = acute["sensor_missing_overview"].astype(int)
acute["sensor_missing_files"] = acute["sensor_missing_files"].astype(int)
acute["sensor_missing_window"] = acute["sensor_missing_window"].astype(int)

print("Long-term rows:", len(lt), "| include_in_yield_model=", int(lt["include_in_yield_model"].sum()))
print("Acute rows:", len(acute), "| include_in_yield_model=", int(acute["include_in_yield_model"].sum()))

lt.head()


Long-term rows: 16 | include_in_yield_model= 16
Acute rows: 48 | include_in_yield_model= 48


,experiment,source_row,cultivar,heat_trt,is_control,set_id,plot_id,paired_plot_id,rotten_count,rotten_weight_g,...,healthy_weight_g,total_count,total_weight_g,pct_rotten_count,pct_rotten_weight,include_in_yield_model,include_in_temp_model,sensor_missing_overview,sensor_missing_files,dataset_lock_version
0,longterm,20,MQ,OTC,0,11,9,13,21,21.74,...,965.46,484,987.20,0.043388,0.022022,1,1,0,0,v1_2026-03-05
1,longterm,13,MQ,OTC,0,12,10,14,11,13.79,...,887.56,459,901.35,0.023965,0.015299,1,1,0,0,v1_2026-03-05
2,longterm,18,MQ,OTC,0,13,11,15,29,45.15,...,693.62,393,738.77,0.073791,0.061115,1,1,0,0,v1_2026-03-05
3,longterm,19,MQ,OTC,0,14,12,16,23,14.44,...,793.71,425,808.15,0.054118,0.017868,1,1,0,0,v1_2026-03-05
4,longterm,16,MQ,Control,1,11,13,9,35,54.15,...,922.96,484,977.11,0.072314,0.055419,1,1,0,0,v1_2026-03-05


In [4]:
acute.head()


,experiment,source_row,cultivar,treatment_raw,heat_level,is_control,replicate,rotten_count,rotten_weight_g,healthy_count,...,pct_rotten_count,pct_rotten_weight,drop_a0,observations,include_in_yield_model,include_in_temp_model,sensor_missing_overview,sensor_missing_files,sensor_missing_window,dataset_lock_version
0,acute,11,MQ,A,A,0,1,81,123.95,344,...,0.190588,0.148512,0,NaN,1,0,0,1,1,v1_2026-03-05
1,acute,26,MQ,AC,A,1,1,22,26.65,509,...,0.041431,0.023581,0,NaN,1,0,0,1,0,v1_2026-03-05
2,acute,9,MQ,A,A,0,2,61,72.99,543,...,0.100993,0.063473,0,NaN,1,0,0,1,1,v1_2026-03-05
3,acute,25,MQ,AC,A,1,2,7,6.80,582,...,0.011885,0.006046,0,NaN,1,0,0,1,0,v1_2026-03-05
4,acute,10,MQ,A,A,0,3,64,74.31,353,...,0.153477,0.096077,0,NaN,1,0,0,1,0,v1_2026-03-05


## 2) Build paired differences (heat - control)


In [5]:
lt_pairs = build_longterm_pairs(lt)
acute_pairs = build_acute_pairs(acute)

print("Long-term pairs:", len(lt_pairs))
print("Acute pairs:", len(acute_pairs))
print("Acute temp-model usable pairs:", int(acute_pairs["temp_model_usable_pair"].sum()))

lt_pairs[["cultivar", "set_id", "healthy_weight_diff", "pct_rotten_count_diff", "pct_rotten_weight_diff"]]


Long-term pairs: 8
Acute pairs: 24
Acute temp-model usable pairs: 6


,cultivar,set_id,healthy_weight_diff,pct_rotten_count_diff,pct_rotten_weight_diff
0,MQ,11,42.50,-0.028926,-0.033397
1,MQ,12,34.43,-0.073081,-0.062757
2,MQ,13,-217.44,-0.030178,-0.000828
3,MQ,14,-363.14,0.003324,-0.004997
4,St,7,190.29,-0.051387,0.005944
5,St,8,-116.10,-0.103811,-0.033783
6,St,9,-40.13,-0.019962,0.000305
7,St,10,-23.72,-0.013387,0.008283


In [6]:
acute_pairs[["cultivar", "heat_level", "replicate", "healthy_weight_diff", "pct_rotten_count_diff", "pct_rotten_weight_diff", "temp_model_usable_pair"]]


,cultivar,heat_level,replicate,healthy_weight_diff,pct_rotten_count_diff,pct_rotten_weight_diff,temp_model_usable_pair
0,MQ,A,1,-392.84,0.149157,0.124932,0
1,MQ,A,2,-40.88,0.089109,0.057427,0
2,MQ,A,3,-243.36,0.098509,0.060153,0
3,MQ,B,1,-497.14,0.073151,0.075322,0
4,MQ,B,2,11.68,0.070091,0.051429,0
5,MQ,B,3,-116.22,0.167197,0.128175,0
6,MQ,C,1,-248.51,0.158296,0.110941,1
7,MQ,C,2,-226.23,0.184891,0.128848,1
8,MQ,C,3,-274.87,0.124188,0.110465,1
9,MQ,D,1,-229.07,0.197152,0.204949,0


## 3) Paired t-test summaries


In [7]:
lt_renamed = lt_pairs.rename(
    columns={
        "healthy_weight_diff": "healthy_weight_g",
        "pct_rotten_count_diff": "pct_rotten_count",
        "pct_rotten_weight_diff": "pct_rotten_weight",
    }
)
acute_renamed = acute_pairs.rename(
    columns={
        "healthy_weight_diff": "healthy_weight_g",
        "pct_rotten_count_diff": "pct_rotten_count",
        "pct_rotten_weight_diff": "pct_rotten_weight",
    }
)

lt_summary = paired_ttest_table(lt_renamed, ["cultivar"], OUTCOMES)
lt_overall = paired_ttest_table(lt_renamed, [], OUTCOMES)
lt_summary = pd.concat([lt_overall, lt_summary], ignore_index=True, sort=False)

acute_summary = paired_ttest_table(acute_renamed, ["cultivar", "heat_level"], OUTCOMES)
acute_overall = paired_ttest_table(acute_renamed, ["heat_level"], OUTCOMES)
acute_summary = pd.concat([acute_overall, acute_summary], ignore_index=True, sort=False)

print("Long-term summary rows:", len(lt_summary))
print("Acute summary rows:", len(acute_summary))

lt_summary


Long-term summary rows: 9
Acute summary rows: 36


,outcome,n_pairs,mean_diff_heat_minus_control,sd_diff,se_diff,t_stat,p_value,cultivar
0,healthy_weight_g,8,-61.663750,170.539352,60.294766,-1.022705,0.340485,NaN
1,pct_rotten_count,8,-0.039676,0.034848,0.012321,-3.220261,0.014648,NaN
2,pct_rotten_weight,8,-0.015153,0.025320,0.008952,-1.692754,0.134336,NaN
3,healthy_weight_g,4,-125.912500,198.936051,99.468026,-1.265859,0.294933,MQ
4,pct_rotten_count,4,-0.032215,0.031348,0.015674,-2.055353,0.132083,MQ
5,pct_rotten_weight,4,-0.025494,0.028749,0.014375,-1.773581,0.174238,MQ
6,healthy_weight_g,4,2.585000,131.448115,65.724057,0.039331,0.971097,St
7,pct_rotten_count,4,-0.047137,0.041261,0.020631,-2.284784,0.106466,St
8,pct_rotten_weight,4,-0.004812,0.019602,0.009801,-0.491026,0.657085,St


In [8]:
acute_summary


,heat_level,outcome,n_pairs,mean_diff_heat_minus_control,sd_diff,se_diff,t_stat,p_value,cultivar
0,A,healthy_weight_g,6,-175.710000,166.119759,67.818108,-2.590901,0.048784,NaN
1,A,pct_rotten_count,6,0.129401,0.091559,0.037379,3.461867,0.018006,NaN
2,A,pct_rotten_weight,6,0.086697,0.095625,0.039039,2.220776,0.077053,NaN
3,B,healthy_weight_g,6,-64.003333,232.157895,94.778064,-0.675297,0.529450,NaN
4,B,pct_rotten_count,6,0.069407,0.053111,0.021683,3.201021,0.023968,NaN
5,B,pct_rotten_weight,6,0.053873,0.044758,0.018272,2.948324,0.031947,NaN
6,C,healthy_weight_g,6,-216.546667,46.541671,19.000558,-11.396858,0.000091,NaN
7,C,pct_rotten_count,6,0.086920,0.081554,0.033294,2.610645,0.047633,NaN
8,C,pct_rotten_weight,6,0.074413,0.049505,0.020210,3.681933,0.014265,NaN
9,D,healthy_weight_g,6,-165.075000,299.745494,122.370586,-1.348976,0.235215,NaN


## 4) Mixed-effects models (primary analysis)


In [9]:
lt_model_df = lt[lt["include_in_yield_model"] == 1].copy()
lt_model_df["set_key"] = lt_model_df["cultivar"].astype(str) + "_" + lt_model_df["set_id"].astype(str)

lt_coef_tables = []
lt_term_tables = []

for outcome in OUTCOMES:
    formula = f"{outcome} ~ C(cultivar) * C(heat_trt)"
    model, optimizer = fit_mixedlm(formula, lt_model_df, "set_key")
    lt_coef_tables.append(mixedlm_coefficient_table(model, outcome, "longterm_mixedlm", optimizer))
    lt_term_tables.append(mixedlm_wald_terms_table(model, outcome, "longterm_mixedlm"))

lt_coef = pd.concat(lt_coef_tables, ignore_index=True)
lt_terms = pd.concat(lt_term_tables, ignore_index=True)

print("Long-term coefficient rows:", len(lt_coef))
print("Long-term term-test rows:", len(lt_terms))

lt_terms.sort_values(["outcome", "p_value"], ascending=[True, True])


Long-term coefficient rows: 15
Long-term term-test rows: 12


/opt/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1906: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1906: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1906: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn

,model_name,outcome,term,statistic,df_constraint,p_value
0,longterm_mixedlm,healthy_weight_g,Intercept,447.014499,1.0,3.219941e-99
1,longterm_mixedlm,healthy_weight_g,C(cultivar),18.932774,1.0,1.354064e-05
2,longterm_mixedlm,healthy_weight_g,C(heat_trt),4.111836,1.0,4.258412e-02
3,longterm_mixedlm,healthy_weight_g,C(cultivar):C(heat_trt),2.141201,1.0,1.433896e-01
4,longterm_mixedlm,pct_rotten_count,Intercept,54.884404,1.0,1.278314e-13
6,longterm_mixedlm,pct_rotten_count,C(heat_trt),4.337525,1.0,3.728109e-02
5,longterm_mixedlm,pct_rotten_count,C(cultivar),2.115172,1.0,1.458456e-01
7,longterm_mixedlm,pct_rotten_count,C(cultivar):C(heat_trt),0.465263,1.0,4.951744e-01
8,longterm_mixedlm,pct_rotten_weight,Intercept,50.903637,1.0,9.701367e-13
9,longterm_mixedlm,pct_rotten_weight,C(cultivar),11.374381,1.0,7.446402e-04


In [10]:
acute_model_df = acute[acute["include_in_yield_model"] == 1].copy()
acute_model_df["pair_id"] = (
    acute_model_df["cultivar"].astype(str)
    + "_"
    + acute_model_df["heat_level"].astype(str)
    + "_"
    + acute_model_df["replicate"].astype(str)
)

acute_coef_tables = []
acute_term_tables = []

for outcome in OUTCOMES:
    formula = f"{outcome} ~ C(cultivar) * C(heat_level) * C(is_control)"
    model, optimizer = fit_mixedlm(formula, acute_model_df, "pair_id")
    acute_coef_tables.append(mixedlm_coefficient_table(model, outcome, "acute_mixedlm", optimizer))
    acute_term_tables.append(mixedlm_wald_terms_table(model, outcome, "acute_mixedlm"))

acute_coef = pd.concat(acute_coef_tables, ignore_index=True)
acute_terms = pd.concat(acute_term_tables, ignore_index=True)

print("Acute coefficient rows:", len(acute_coef))
print("Acute term-test rows:", len(acute_terms))

acute_terms.sort_values(["outcome", "p_value"], ascending=[True, True])


Acute coefficient rows: 51
Acute term-test rows: 24


/opt/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1906: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1906: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1906: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn

,model_name,outcome,term,statistic,df_constraint,p_value
0,acute_mixedlm,healthy_weight_g,Intercept,120.210488,1.0,5.689216e-28
3,acute_mixedlm,healthy_weight_g,C(is_control),5.059529,1.0,2.449089e-02
2,acute_mixedlm,healthy_weight_g,C(heat_level),3.238708,3.0,3.562649e-01
7,acute_mixedlm,healthy_weight_g,C(cultivar):C(heat_level):C(is_control),3.098351,3.0,3.767086e-01
4,acute_mixedlm,healthy_weight_g,C(cultivar):C(heat_level),2.486318,3.0,4.777688e-01
5,acute_mixedlm,healthy_weight_g,C(cultivar):C(is_control),0.496310,1.0,4.811260e-01
1,acute_mixedlm,healthy_weight_g,C(cultivar),0.110596,1.0,7.394665e-01
6,acute_mixedlm,healthy_weight_g,C(heat_level):C(is_control),0.855062,3.0,8.362567e-01
8,acute_mixedlm,pct_rotten_count,Intercept,19.441427,1.0,1.037324e-05
11,acute_mixedlm,pct_rotten_count,C(is_control),5.565990,1.0,1.831266e-02


## 5) Preview report content


In [11]:
report_lines = build_report_lines(
    lt,
    acute,
    lt_pairs,
    acute_pairs,
    lt_summary,
    acute_summary,
    lt_terms,
    acute_terms,
)
print("\n".join(report_lines[:40]))


# Yield Analysis Report

- Generated at: 2026-03-19T12:08:52
- Outcome coding for paired summaries: all differences are `heat - control`.
- Primary model: mixed-effects model on row-level data.
- Long-term mixed model: fixed `cultivar * heat_trt`, random intercept `(1|cultivar:set_id)`.
- Acute mixed model: fixed `cultivar * heat_level * is_control`, random intercept `(1|cultivar:heat_level:replicate)`.

## Inputs
- /Users/liyuang/Desktop/STAT628/installment3/cleaned_data/longterm_yield_analysis_locked.csv
- /Users/liyuang/Desktop/STAT628/installment3/cleaned_data/acute_yield_analysis_locked.csv

## Key counts
- Long-term rows in yield model: 16
- Long-term pairs: 8
- Acute rows in yield model: 48
- Acute pairs: 24
- Acute pairs usable for later temperature models: 6

## Paired-summary quick check
- Long-term healthy weight summary:
  - overall: mean diff=-61.664, p=0.34048493770273314
  - MQ: mean diff=-125.912, p=0.294932692418296
  - St: mean diff=2.585, p=0.9710974326370292
- Long-

## 6) Final step: write all outputs to files


In [12]:
RESULT_DIR.mkdir(parents=True, exist_ok=True)

lt_summary.to_csv(LT_SUMMARY_PATH, index=False)
acute_summary.to_csv(ACUTE_SUMMARY_PATH, index=False)

lt_coef.to_csv(LT_COEF_PATH, index=False)
lt_terms.to_csv(LT_WALD_PATH, index=False)
acute_coef.to_csv(ACUTE_COEF_PATH, index=False)
acute_terms.to_csv(ACUTE_WALD_PATH, index=False)

REPORT_PATH.write_text("\n".join(report_lines), encoding="utf-8")

print("Wrote files:")
for p in [
    LT_SUMMARY_PATH,
    LT_COEF_PATH,
    LT_WALD_PATH,
    ACUTE_SUMMARY_PATH,
    ACUTE_COEF_PATH,
    ACUTE_WALD_PATH,
    REPORT_PATH,
]:
    print("-", p)


Wrote files:
- /Users/liyuang/Desktop/STAT628/installment3/results/yield/yield_longterm_summary.csv
- /Users/liyuang/Desktop/STAT628/installment3/results/yield/yield_longterm_mixedlm_coefficients.csv
- /Users/liyuang/Desktop/STAT628/installment3/results/yield/yield_longterm_mixedlm_wald_terms.csv
- /Users/liyuang/Desktop/STAT628/installment3/results/yield/yield_acute_summary.csv
- /Users/liyuang/Desktop/STAT628/installment3/results/yield/yield_acute_mixedlm_coefficients.csv
- /Users/liyuang/Desktop/STAT628/installment3/results/yield/yield_acute_mixedlm_wald_terms.csv
- /Users/liyuang/Desktop/STAT628/installment3/results/yield/yield_analysis_report.md
